# Chapter 31 — Short-Rate Models

Every worked example in the chapter, recomputed. The parameters are the chapter's, and
each figure is printed beside the one the book prints, so a number that has drifted is
visible rather than merely wrong.

| Section | What it reproduces | Example |
|---|---|---|
| 1 | Vasicek's curve from three starting rates | 31.1 |
| 2 | CIR against Vasicek at matched volatility | 31.2 |
| 3 | What $a$ and $\sigma$ separately control | 31.3 |
| 4 | The Hull–White machinery fitted to a curve | §31.6 |
| 5 | A cap and a swaption in closed form | 31.4 |
| 6 | The same caplet three ways | 31.5 |
| 7 | The forward is not the $Q$-expectation | 31.6 |
| 8 | Building the tree, step by step | 31.7, §31.8 |
| 9 | One option with five dates | 31.8 |
| 10 | Every European matched, the Bermudan not | 31.9 |
| 11 | Z-spread against option-adjusted spread | 31.10 |
| 12 | What perfect correlation costs | 31.11 |
| 13 | How often the model says rates are below zero | 31.12 |

Almost everything here is deterministic and reproduces exactly. The one simulation, in
Section 6, draws from a distribution that is known in closed form, so it carries a
standard error but no discretization bias. The whole notebook runs in about twenty
seconds.

In [1]:
import numpy as np
from math import ceil, exp, log, pi, sqrt
from scipy.optimize import brentq
from scipy.stats import norm

N = norm.cdf


def show(label, got, printed, tol=6e-5):
    """Print a computed value beside the one the chapter prints."""
    flag = "ok" if abs(got - printed) <= tol * max(1.0, abs(printed)) else "DIFFERS"
    print("%-46s %14.6f   book %13.6f   %s" % (label, got, printed, flag))

## 1. Vasicek's curve from three starting rates — Example 31.1

Three starting short rates, one set of parameters, and the curves they produce. All three
converge to the same long rate, and that rate sits *below* $b$: the $-\sigma^2/(2a^2)$ is
a convexity term, and it is the first sign that an equilibrium model's shape is not a free
choice.

In [2]:
A_V, B_V, SIG_V = 0.10, 0.05, 0.01


def vasicek_zero(r0, T, a=A_V, b=B_V, sigma=SIG_V):
    """Vasicek's continuously compounded zero rate, from the affine bond price."""
    T = np.asarray(T, dtype=float)
    B = (1.0 - np.exp(-a * T)) / a
    lnA = (B - T) * (a * a * b - 0.5 * sigma ** 2) / a ** 2 - sigma ** 2 * B * B / (4.0 * a)
    return (B * r0 - lnA) / T


MATS = (0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 30.0)
BOOK_V = ((2.073, 5.000, 7.926), (2.144, 4.998, 7.853), (2.275, 4.994, 7.713),
          (2.610, 4.971, 7.332), (3.020, 4.916, 6.812), (3.513, 4.810, 6.107),
          (3.783, 4.734, 5.684))
print("%9s %9s %9s %9s     %s" % ("maturity", "r0 = 2%", "r0 = 5%", "r0 = 8%", "book"))
for T, want in zip(MATS, BOOK_V):
    got = [100 * float(vasicek_zero(r0, T)) for r0 in (0.02, 0.05, 0.08)]
    flag = "ok" if max(abs(g - w) for g, w in zip(got, want)) < 5e-4 + 1e-9 else "DIFFERS"
    print("%8.1fy %8.3f%% %8.3f%% %8.3f%%     %s" % (T, got[0], got[1], got[2], flag))
show("R_infinity = b - sigma^2/(2a^2), %",
     100 * (B_V - SIG_V ** 2 / (2 * A_V ** 2)), 4.50, tol=5e-4)

 maturity   r0 = 2%   r0 = 5%   r0 = 8%     book
     0.5y    2.073%    5.000%    7.926%     ok
     1.0y    2.144%    4.998%    7.853%     ok
     2.0y    2.275%    4.994%    7.713%     ok
     5.0y    2.610%    4.971%    7.332%     ok
    10.0y    3.020%    4.916%    6.812%     ok
    20.0y    3.513%    4.810%    6.107%     ok
    30.0y    3.783%    4.734%    5.684%     ok
R_infinity = b - sigma^2/(2a^2), %                   4.500000   book      4.500000   ok


### The third shape, and it is barely available

Vasicek can produce a humped curve, but only from a starting rate inside a band 75 basis
points wide, and the largest hump anywhere in that band is about ten basis points. Both
facts are worth computing rather than asserting, because the band is the whole answer to
"can this model make a humped curve?"

The scan below measures a hump as the peak's height over the *higher* of the two ends, out
to six hundred years. A curve that merely rises to a plateau is not humped.

In [3]:
HGRID = np.arange(1, 12001) * 0.05          # out to 600 years: a hump has to be a peak


def hump(zero, r0):
    """Height of the peak over the higher of the two ends, in basis points, and where."""
    y = zero(r0, HGRID)
    k = int(np.argmax(y))
    if k in (0, len(y) - 1):
        return 0.0, None
    far = 2 * y[-1] - y[len(y) // 2 - 1]    # 600 and 300 years, extrapolated in 1/T to R_infinity
    return 10000 * (y[k] - max(r0, far)), HGRID[k]


def largest_hump(zero, lo, hi):
    """Scan the humped band coarsely, then refine around the best starting rate."""
    best = lo
    for step in (5e-5, 5e-7):
        grid = np.arange(lo, hi, step)
        best = max(grid, key=lambda r: hump(zero, r)[0])
        lo, hi = best - step, best + step
    h, peak = hump(zero, best)
    return h, best, peak


V_LO = B_V - 3 * SIG_V ** 2 / (4 * A_V ** 2)
show("humped band, lower edge, %", 100 * V_LO, 4.25, tol=5e-4)
show("humped band, width, bp", 10000 * (B_V - V_LO), 75.0, tol=5e-4)
h_v, r_v, t_v = largest_hump(vasicek_zero, V_LO, B_V)
show("largest hump Vasicek can make, bp", h_v, 10.0, tol=3e-2)
show("the r0 that produces it, %", 100 * r_v, 4.50, tol=1e-3)
show("and that r0 is R_infinity, %", 100 * r_v,
     100 * (B_V - SIG_V ** 2 / (2 * A_V ** 2)), tol=1e-3)
show("maturity of the peak, years", t_v, 12.5, tol=1e-2)

humped band, lower edge, %                           4.250000   book      4.250000   ok
humped band, width, bp                              75.000000   book     75.000000   ok
largest hump Vasicek can make, bp                   10.181604   book     10.000000   ok
the r0 that produces it, %                           4.500000   book      4.500000   ok
and that r0 is R_infinity, %                         4.500000   book      4.500000   ok
maturity of the peak, years                         12.550000   book     12.500000   ok


## 2. CIR against Vasicek at matched volatility — Example 31.2

The two models are comparable only if their volatilities are. Vasicek's diffusion is
$\sigma$ and CIR's is $\sigma\sqrt{r}$, so they agree at $r = b$ when CIR's $\sigma$ is
$0.01/\sqrt{0.05} = 4.47\%$. The Feller condition then holds comfortably.

The second table is the difference between the two curves. It is what positivity costs,
and at short maturities it costs nothing at all.

In [4]:
SIG_CIR = 0.0447
GAMMA = sqrt(A_V ** 2 + 2 * SIG_CIR ** 2)


def cir_zero(r0, tau, a=A_V, b=B_V, sigma=SIG_CIR):
    """CIR's zero rate. The same affine shape, a different B and A."""
    tau = np.asarray(tau, dtype=float)
    g = sqrt(a * a + 2 * sigma * sigma)
    e = np.exp(g * tau)
    den = (g + a) * (e - 1.0) + 2.0 * g
    B = 2.0 * (e - 1.0) / den
    lnA = (2.0 * a * b / sigma ** 2) * (log(2.0 * g) + (a + g) * tau / 2.0 - np.log(den))
    return (B * r0 - lnA) / tau


show("sigma_CIR matching 1% at r = b", 0.01 / sqrt(B_V), 0.0447, tol=5e-4)
show("Feller: 2ab", 2 * A_V * B_V, 0.0100, tol=5e-4)
show("Feller: sigma^2", SIG_CIR ** 2, 0.0020, tol=2e-3)

BOOK_C = ((2.074, 5.000, 7.926), (2.144, 4.998, 7.852), (2.278, 4.994, 7.710),
          (2.626, 4.971, 7.317), (3.059, 4.918, 6.777), (3.586, 4.821, 6.057),
          (3.869, 4.757, 5.645))
print()
print("%9s %9s %9s %9s     %s" % ("maturity", "r0 = 2%", "r0 = 5%", "r0 = 8%", "book"))
for T, want in zip(MATS, BOOK_C):
    got = [100 * float(cir_zero(r0, T)) for r0 in (0.02, 0.05, 0.08)]
    flag = "ok" if max(abs(g - w) for g, w in zip(got, want)) < 5e-4 + 1e-9 else "DIFFERS"
    print("%8.1fy %8.3f%% %8.3f%% %8.3f%%     %s" % (T, got[0], got[1], got[2], flag))

print()
print("%9s %9s %9s %9s" % ("maturity", "r0 = 2%", "r0 = 5%", "r0 = 8%"))
for T in (1, 2, 5, 10, 20, 30):
    print("%8dy %9.1f %9.1f %9.1f"
          % (T, 10000 * float(cir_zero(0.02, T) - vasicek_zero(0.02, T)),
             10000 * float(cir_zero(0.05, T) - vasicek_zero(0.05, T)),
             10000 * float(cir_zero(0.08, T) - vasicek_zero(0.08, T))))
print("CIR less Vasicek, in basis points")

sigma_CIR matching 1% at r = b                       0.044721   book      0.044700   ok
Feller: 2ab                                          0.010000   book      0.010000   ok
Feller: sigma^2                                      0.001998   book      0.002000   ok

 maturity   r0 = 2%   r0 = 5%   r0 = 8%     book
     0.5y    2.074%    5.000%    7.926%     ok
     1.0y    2.144%    4.998%    7.852%     ok
     2.0y    2.278%    4.994%    7.710%     ok
     5.0y    2.626%    4.971%    7.317%     ok
    10.0y    3.059%    4.918%    6.777%     ok
    20.0y    3.586%    4.821%    6.057%     ok
    30.0y    3.869%    4.757%    5.645%     ok

 maturity   r0 = 2%   r0 = 5%   r0 = 8%
       1y       0.1       0.0      -0.1
       2y       0.3       0.0      -0.3
       5y       1.5       0.0      -1.5
      10y       4.0       0.2      -3.6
      20y       7.3       1.2      -5.0
      30y       8.6       2.3      -3.9
CIR less Vasicek, in basis points


### CIR's hump

The same scan, on CIR. Its band is narrower than Vasicek's and its largest hump is smaller,
which is the point: the constraint that keeps rates positive also removes shapes.

CIR's band has no simple closed form, so its lower edge is found by root-finding on the
hump height rather than written down.

In [5]:
show("CIR's long rate 2ab/(gamma + a), %", 100 * 2 * A_V * B_V / (GAMMA + A_V), 4.58, tol=5e-4)
C_LO = brentq(lambda r: hump(cir_zero, r)[0] - 1e-9, 0.040, 0.0499, xtol=1e-9)
show("CIR's humped band, lower edge, %", 100 * C_LO, 4.40, tol=1e-3)
show("CIR's humped band, width, bp", 10000 * (B_V - C_LO), 60.1, tol=2e-3)
show("narrower than Vasicek's by, bp",
     10000 * (B_V - V_LO) - 10000 * (B_V - C_LO), 14.9, tol=1e-2)
h_c, r_c, t_c = largest_hump(cir_zero, C_LO, B_V)
show("largest hump CIR can make, bp", h_c, 8.0, tol=3e-2)
show("the r0 that produces it, %", 100 * r_c, 4.58, tol=1e-3)
show("maturity of the peak, years", t_c, 11.1, tol=2e-2)

CIR's long rate 2ab/(gamma + a), %                   4.580738   book      4.580000   ok
CIR's humped band, lower edge, %                     4.398788   book      4.400000   ok
CIR's humped band, width, bp                        60.121224   book     60.100000   ok
narrower than Vasicek's by, bp                      14.878776   book     14.900000   ok
largest hump CIR can make, bp                        7.791863   book      8.000000   ok
the r0 that produces it, %                           4.580738   book      4.580000   ok
maturity of the peak, years                         11.150000   book     11.100000   ok


## 3. What $a$ and $\sigma$ separately control — Example 31.3

Forward-rate volatility under Hull–White is $\sigma e^{-aT}$, which factors cleanly:
$\sigma$ sets the level and $a$ sets the shape. At $a = 0.01$ a thirty-year forward is
three-quarters as volatile as a one-year forward; at $a = 0.20$ it has essentially no
volatility at all.

In [6]:
A_HW, SIG_HW, L = 0.08, 0.01, 100.0

print("%9s %9s %9s %9s %9s" % ("maturity", "a = 0.01", "a = 0.05", "a = 0.10", "a = 0.20"))
for T in (1, 2, 5, 10, 20, 30):
    print("%8dy %9.1f %9.1f %9.1f %9.1f"
          % (T, *(100.0 * exp(-a * T) for a in (0.01, 0.05, 0.10, 0.20))))
print("Forward-rate volatility in basis points, at sigma = 1%")

 maturity  a = 0.01  a = 0.05  a = 0.10  a = 0.20
       1y      99.0      95.1      90.5      81.9
       2y      98.0      90.5      81.9      67.0
       5y      95.1      77.9      60.7      36.8
      10y      90.5      60.7      36.8      13.5
      20y      81.9      36.8      13.5       1.8
      30y      74.1      22.3       5.0       0.2
Forward-rate volatility in basis points, at sigma = 1%


## 4. The Hull–White machinery, fitted to a curve

Everything from here on needs $A(t,T)$, and $A(t,T)$ depends on which curve the model has
been fitted to. Two curves appear in the chapter: the flat 4% curve of Example 31.4, and
the market curve $R(0,T) = 4.2\% - 1.2\%e^{-0.25T}$ that Sections 31.7 onward use. One small
class carries a curve together with the $\ln A(t,T)$, $P(t,T)$ and $\theta(t)$ that follow
from it, so neither has to be written twice.

In [7]:
def B_hw(t, T, a=A_HW):
    return (1.0 - np.exp(-a * (T - t))) / a


class Curve:
    """Today's curve, and the Hull-White quantities fitted to it."""

    def __init__(self, lnP, f0):
        self.lnP, self.f0 = lnP, f0

    def P(self, T):
        return np.exp(self.lnP(np.asarray(T, dtype=float)))

    def lnA(self, t, T):
        """Section 31.6's ln A(t,T): what makes the model reproduce this curve."""
        return (self.lnP(T) - self.lnP(t) + B_hw(t, T) * self.f0(t)
                - SIG_HW ** 2 / (4 * A_HW) * (1 - np.exp(-2 * A_HW * t)) * B_hw(t, T) ** 2)

    def P_fwd(self, t, T, r):
        """Equation (31.2): the time-t bond price when the short rate is r."""
        return np.exp(self.lnA(t, T) - B_hw(t, T) * r)

    def theta(self, t, h=1e-6):
        """Section 31.6's theta(t) = f_T(0,t) + a f(0,t) + sigma^2 (1 - e^{-2at})/(2a)."""
        return ((self.f0(t + h) - self.f0(t - h)) / (2 * h) + A_HW * self.f0(t)
                + SIG_HW ** 2 / (2 * A_HW) * (1 - np.exp(-2 * A_HW * t)))


FLAT = Curve(lambda T: -0.04 * np.asarray(T, dtype=float), lambda t: 0.04 + 0.0 * t)

# Sections 31.7 onward use the market curve R(T) = 4.2% - 1.2% exp(-0.25T), whose
# instantaneous forward rate is f(0,T) = R(T) + T R'(T).
MKT = Curve(lambda T: -(0.042 - 0.012 * np.exp(-0.25 * np.asarray(T, dtype=float)))
            * np.asarray(T, dtype=float),
            lambda t: 0.042 - 0.012 * np.exp(-0.25 * t) + 0.003 * t * np.exp(-0.25 * t))

print("%-10s %14s" % ("maturity", "P(0,T)"))
for T in (1, 2, 3, 5, 7, 10):
    print("%9dy %14.8f" % (T, MKT.P(T)))
show("f(0,0), which is r(0), %", 100 * float(MKT.f0(0.0)), 3.0, tol=1e-4)


def sigma_P(T, S, a=A_HW, sigma=SIG_HW):
    """Equation (31.8): the forward bond price's accumulated volatility."""
    return sigma / a * (1 - exp(-a * (S - T))) * sqrt((1 - exp(-2 * a * T)) / (2 * a))


def zbp(curve, T, S, X):
    """Equation (31.10): a European put on the S-bond, expiring at T, struck at X."""
    sp = sigma_P(T, S)
    h = np.log(curve.P(S) / (X * curve.P(T))) / sp + sp / 2
    return X * curve.P(T) * N(sp - h) - curve.P(S) * N(-h)

maturity           P(0,T)
        1y     0.96787301
        2y     0.93291306
        3y     0.89673508
        5y     0.82463888
        7y     0.75623509
       10y     0.66355084
f(0,0), which is r(0), %                             3.000000   book      3.000000   ok


## 5. A cap in closed form — Example 31.4

A two-year semiannual cap at 4% on the flat curve. Each caplet is
$L(1 + \tau K)$ puts on the period's end bond, struck at $1/(1+\tau K)$, and each put is
equation (31.10).

The caplets rise in value with their expiration although the forward rate is identical in
each, because $\sigma_P$ grows with the option's life — and that growth is bounded, which
is mean reversion at work.

In [8]:
TAU_CAP, K_CAP = 0.5, 0.04
X_CAP = 1.0 / (1.0 + TAU_CAP * K_CAP)
show("forward rate each period, %",
     100 * float(FLAT.P(0.5) / FLAT.P(1.0) - 1.0) / TAU_CAP, 4.040268)

print()
print("%-14s %10s %13s" % ("caplet", "sigma_P", "value"))
cap = 0.0
for T, S in ((0.5, 1.0), (1.0, 1.5), (1.5, 2.0)):
    v = L * (1.0 + TAU_CAP * K_CAP) * float(zbp(FLAT, T, S, X_CAP))
    cap += v
    print("[%.1f, %.1f]      %10.2f %13.6f" % (T, S, 10000 * sigma_P(T, S), v))
show("two-year cap", cap, 0.555017)

forward rate each period, %                          4.040268   book      4.040268   ok

caplet            sigma_P         value
[0.5, 1.0]           33.98      0.142743
[1.0, 1.5]           47.12      0.190218
[1.5, 2.0]           56.60      0.222055
two-year cap                                         0.555017   book      0.555017   ok


### And a swaption, by Jamshidian

A European option on a coupon bond is a portfolio of options on the zeros, once the
threshold rate $r^{*}$ that puts the coupon bond at the strike has been found. Almost all
the value sits in the final strip, which carries the principal.

In [9]:
T_EXP, PAY = 1.0, (2.0, 3.0, 4.0)
ann = sum(float(FLAT.P(t)) for t in PAY)
par = float(FLAT.P(T_EXP) - FLAT.P(PAY[-1])) / ann
show("annuity", ann, 2.662181)
show("par swap rate, %", 100 * par, 4.081077)

COUPONS = [par, par, par + 1.0]


def coupon_bond(r):
    """The underlying swap's fixed leg, seen from the expiration date."""
    return sum(c * FLAT.P_fwd(T_EXP, t, r) for c, t in zip(COUPONS, PAY))


r_star = brentq(lambda r: coupon_bond(r) - 1.0, -0.5, 0.5, xtol=1e-14)
show("r* solving the Jamshidian threshold, %", 100 * r_star, 3.987895)

print()
print("%-10s %11s %11s %13s" % ("maturity", "c_i", "X_i", "value"))
swaption = 0.0
for c, t in zip(COUPONS, PAY):
    X_i = float(FLAT.P_fwd(T_EXP, t, r_star))
    v = L * c * float(zbp(FLAT, T_EXP, t, X_i))
    swaption += v
    print("%9.0fy %11.6f %11.6f %13.6f" % (t, c, X_i, v))
show("one-into-three payer swaption", swaption, 0.946713)

annuity                                              2.662181   book      2.662181   ok
par swap rate, %                                     4.081077   book      4.081077   ok
r* solving the Jamshidian threshold, %               3.987895   book      3.987895   ok

maturity           c_i         X_i         value
        2y    0.040811    0.960860      0.014025
        3y    0.040811    0.923177      0.025775
        4y    1.040811    0.886915      0.906913
one-into-three payer swaption                        0.946713   book      0.946713   ok


### Both again, by integration

The independent check. Integrating the payoff directly over the distribution of $r(T)$
shares no formula with Jamshidian beyond the bond price itself — and it only agrees under
the $T$-forward measure, where the mean of $r(T)$ is $f(0,T)$ exactly. Under the
risk-neutral measure the mean carries a further
$(\sigma^2/(2a^2))(1-e^{-aT})^2$, and the last line shows what that one slip costs.

In [10]:
def integrate(T, payoff, risk_neutral=False, n=40001):
    """P(0,T) E^T[payoff(r(T))]. Under Q^T the mean of r(T) is f(0,T) exactly."""
    mu = 0.04 + (SIG_HW ** 2 / (2 * A_HW ** 2) * (1 - exp(-A_HW * T)) ** 2
                 if risk_neutral else 0.0)
    sd = sqrt(SIG_HW ** 2 / (2 * A_HW) * (1 - exp(-2 * A_HW * T)))
    r = np.linspace(mu - 9 * sd, mu + 9 * sd, n)
    w = np.ones(n)
    w[1:-1:2], w[2:-1:2] = 4.0, 2.0
    dens = np.exp(-0.5 * ((r - mu) / sd) ** 2) / (sd * sqrt(2 * pi))
    return float(FLAT.P(T)) * float(np.sum(w * payoff(r) * dens)) * (r[1] - r[0]) / 3.0


direct = integrate(T_EXP, lambda r: L * np.maximum(1.0 - coupon_bond(r), 0.0))
show("the swaption again, integrated under Q^T", direct, 0.946713)
print("    the two routes differ by %.2e" % abs(swaption - direct))

wrong = integrate(T_EXP, lambda r: L * np.maximum(1.0 - coupon_bond(r), 0.0),
                  risk_neutral=True)
show("the same integral under Q instead", wrong, 0.952366)
show("what that one measure slip costs, %", 100 * (wrong / direct - 1.0), 0.6, tol=5e-2)

the swaption again, integrated under Q^T             0.946713   book      0.946713   ok
    the two routes differ by 1.00e-09
the same integral under Q instead                    0.952366   book      0.952366   ok
what that one measure slip costs, %                  0.597170   book      0.600000   ok


## 6. The same caplet three ways — Example 31.5

A caplet on the simple rate over $[2.0, 2.5]$, struck at 4%, on the market curve.

**(a) Under the forward measure.** The forward rate is a martingale, so there is no drift
to model, and the caplet is $L(1+\tau K)$ puts on the 2.5-year bond. That bond is
lognormal with a deterministic volatility, which is exactly Black's setting, so the closed
form is exact rather than an approximation. Both routes to it are shown.

In [11]:
T1, T2, K = 2.0, 2.5, 0.04
TAU = T2 - T1
P_T1, P_T2 = float(MKT.P(T1)), float(MKT.P(T2))
F0 = (P_T1 / P_T2 - 1.0) / TAU
show("forward rate over [2.0, 2.5], %", 100 * F0, 3.9381, tol=2e-4)

sp = sigma_P(T1, T2)
show("forward bond volatility, bp", 10000 * sp, 64.12, tol=1e-4)
d1 = log((1 + TAU * F0) / (1 + TAU * K)) / sp + sp / 2
caplet = L * P_T2 * ((1 + TAU * F0) * N(d1) - (1 + TAU * K) * N(d1 - sp))
show("(a) the caplet under the forward measure", caplet, 0.22478353)

X_B = 1.0 / (1.0 + TAU * K)
h = log(P_T2 / (X_B * P_T1)) / sp + sp / 2
show("(a) again, as L(1 + tau K) bond puts",
     L * (1 + TAU * K) * (X_B * P_T1 * N(sp - h) - P_T2 * N(-h)), 0.22478353)

forward rate over [2.0, 2.5], %                      3.938048   book      3.938100   ok
forward bond volatility, bp                         64.122465   book     64.120000   ok
(a) the caplet under the forward measure             0.224784   book      0.224784   ok
(a) again, as L(1 + tau K) bond puts                 0.224784   book      0.224784   ok


### (b) Under the risk-neutral measure

Now the numeraire is the money market account, so the payoff has to be discounted along
each path of $r$. There is no need to step through time: under $Q$ the pair
$\left(r(T_1), \int_0^{T_1} r\,ds\right)$ is jointly Gaussian with moments known in closed
form, so the draw is exact and carries a standard error but no discretization bias.

The first line printed is worth as much as the answer. The mean of $e^{-I}$ must come back
as $P(0,2)$ — that identity is what fixes $E^Q[I]$, and it says the expected rolled-up money
market account is not $1/P(0,T)$ but carries a variance term.

The book's 0.22498411 came from two hundred time steps on two million antithetic paths; this
draw is exact in time, so it agrees with (a) to its own standard error rather than with the
book's digits.

In [12]:
# Under Q both r(T1) and I = int_0^T1 r ds are Gaussian and their joint law is known, so
# the simulation needs no time grid and carries no discretization bias.
var_r = SIG_HW ** 2 * (1 - exp(-2 * A_HW * T1)) / (2 * A_HW)
var_I = (SIG_HW / A_HW) ** 2 * (T1 - 2 * (1 - exp(-A_HW * T1)) / A_HW
                                + (1 - exp(-2 * A_HW * T1)) / (2 * A_HW))
cov_rI = (SIG_HW ** 2 / A_HW) * ((1 - exp(-A_HW * T1)) / A_HW
                                 - (1 - exp(-2 * A_HW * T1)) / (2 * A_HW))
mean_r = float(MKT.f0(T1)) + SIG_HW ** 2 / (2 * A_HW ** 2) * (1 - exp(-A_HW * T1)) ** 2
mean_I = -float(MKT.lnP(T1)) + var_I / 2      # which is E[e^{-I}] = P(0,T1), rearranged

rng = np.random.default_rng(31)
z = rng.standard_normal((1_000_000, 2))
z = np.concatenate([z, -z])                   # antithetic, so two million in all
r_T1 = mean_r + sqrt(var_r) * z[:, 0]
I = mean_I + cov_rI / sqrt(var_r) * z[:, 0] + sqrt(var_I - cov_rI ** 2 / var_r) * z[:, 1]

show("E^Q[e^{-I}], which must be P(0,2)", float(np.exp(-I).mean()), P_T1, tol=1e-5)
bond = MKT.P_fwd(T1, T2, r_T1)
payoff = L * np.maximum(1.0 / bond - (1.0 + TAU * K), 0.0)
pv = np.exp(-I) * bond * payoff
se = pv.std(ddof=1) / sqrt(pv.size)
show("(b) the same caplet, simulated under Q", float(pv.mean()), caplet, tol=5e-3)
print("    standard error %.8f, so the gap is %.2f of them"
      % (se, abs(pv.mean() - caplet) / se))

E[e^{-I}], which must be P(0,2)                      0.932913   book      0.932913   ok


(b) the same caplet, simulated under Q               0.224877   book      0.224784   ok


    standard error 0.00023569, so the gap is 0.40 of them


### (c) The error the measure change exists to prevent

Take the expected payoff under $Q$ and discount it at today's zero rate — treating the
payoff and the discount factor as independent. Under $Q$ the forward bond ratio is not a
martingale, so this is computable in closed form: it is (a)'s own formula with the forward
scaled by $e^{\mu}$, where $\mu$ is the drift the change of numeraire removes.

The error is the covariance between the payoff and the discount factor. A caplet pays when
rates are high, and when rates are high the discount factor is small, so ignoring the
correlation overstates the value.

In [13]:
# (c) discounts E^Q[payoff] at today's zero rate. Under Q the forward bond ratio is not a
# martingale: it drifts by int_0^T1 nu(t) sigma_B(t,T2) dt, with nu the ratio's own
# volatility, so (a)'s formula applies with (1 + tau F0) replaced by (1 + tau F0)e^{mu}.
mu_Q = ((SIG_HW / A_HW) ** 2 * (1 - exp(-A_HW * TAU))
        * ((1 - exp(-A_HW * T1)) / A_HW
           - exp(-A_HW * TAU) * (1 - exp(-2 * A_HW * T1)) / (2 * A_HW)))
fwd_Q = (1 + TAU * F0) * exp(mu_Q)
d1c = log(fwd_Q / (1 + TAU * K)) / sp + sp / 2
naive = L * P_T2 * (fwd_Q * N(d1c) - (1 + TAU * K) * N(d1c - sp))
show("(c) E^Q[payoff] discounted at today's rate", naive, 0.23044665)
show("(c) the same, from the paths above", float(payoff.mean()) * P_T2, naive, tol=5e-3)
show("(c) error from the dropped covariance, %", 100 * (naive / caplet - 1.0), 2.52, tol=4e-3)

(c) E^Q[payoff] discounted at today's rate           0.230447   book      0.230447   ok
(c) the same, from the paths above                   0.230544   book      0.230447   ok
(c) error from the dropped covariance, %             2.519365   book      2.520000   ok


## 7. The forward is not the $Q$-expectation — Example 31.6

A forward price is an expectation under the $T$-forward measure and is not one under the
risk-neutral measure. The gap has two parts. The first survives at $\rho = 0$ and is the discount factor's own
convexity, $e^{\mathrm{Var}^Q(\int_0^T r\,ds)}$, the variance term that appeared a moment ago;
the second moves with the correlation between the asset and rates. Together they are the
covariance between the asset and the discount factor.

In [14]:
S0, SIG_S, T_EQ = 100.0, 0.25, 5.0
P_5 = float(MKT.P(T_EQ))
show("P(0,5) from the curve", P_5, 0.824639, tol=1e-5)
show("the forward price S0/P(0,T)", S0 / P_5, 121.2652, tol=1e-5)

B_T = float(B_hw(0.0, T_EQ))
cov_unit = (SIG_HW / A_HW) * (T_EQ - B_T)
var_Y = (SIG_HW / A_HW) ** 2 * (T_EQ - 2 * B_T + (1 - exp(-2 * A_HW * T_EQ)) / (2 * A_HW))
print()
print("%8s %14s %10s     %s" % ("rho", "E^Q[S_T]/F", "gap", "book"))
for rho, want in ((-0.50, -1.056), (-0.25, -0.374), (0.00, 0.313),
                  (0.25, 1.004), (0.50, 1.700)):
    ratio = exp(var_Y + SIG_S * rho * cov_unit)
    flag = "ok" if abs(100 * (ratio - 1) - want) < 6e-3 else "DIFFERS"
    print("%+8.2f %14.6f %9.3f%%     %s" % (rho, ratio, 100 * (ratio - 1), flag))
show("at rho = 0, the convexity factor alone", exp(var_Y), 1.003125)

P(0,5) from the curve                                0.824639   book      0.824639   ok
the forward price S0/P(0,T)                        121.265202   book    121.265200   ok

     rho     E^Q[S_T]/F        gap     book
   -0.50       0.989442    -1.056%     ok
   -0.25       0.996260    -0.374%     ok
   +0.00       1.003125     0.313%     ok
   +0.25       1.010037     1.004%     ok
   +0.50       1.016997     1.700%     ok
at rho = 0, the convexity factor alone               1.003125   book      1.003125   ok


## 8. Building the tree, step by step — Example 31.7

Stage one builds a tree for $dx = -ax\,dt + \sigma\,dW$, the process with $\theta(t)$ removed, which reverts to zero. The spacing
$\Delta x = \sigma\sqrt{3\Delta t}$ makes the three branching probabilities come out as
$1/6$, $2/3$, $1/6$ at the center, and the middle one goes negative once
$a j \Delta t > \sqrt{2/3}$ — which is where $j_{\max}$ comes from and why the outer nodes
branch differently.

Note the two switched rows at $j = \pm 3$: they branch back toward the center rather than
outward, which is the tree refusing to widen.

In [15]:
A0, SIG0 = 0.08, 0.01


def branch(j, a, dt, jmax):
    """The three probabilities at level j, and the levels they branch to."""
    u = a * j * dt
    if j == jmax:
        return ((7.0 / 6 + (u * u - 3 * u) / 2, -1.0 / 3 - u * u + 2 * u,
                 1.0 / 6 + (u * u - u) / 2), (0, -1, -2))
    if j == -jmax:
        return ((1.0 / 6 + (u * u + u) / 2, -1.0 / 3 - u * u - 2 * u,
                 7.0 / 6 + (u * u + 3 * u) / 2), (2, 1, 0))
    return ((1.0 / 6 + (u * u - u) / 2, 2.0 / 3 - u * u,
             1.0 / 6 + (u * u + u) / 2), (1, 0, -1))


def build(a, sigma, dt, n):
    """Stage one is the geometry; stage two solves alpha(i) off the Arrow-Debreu prices."""
    dx = sigma * sqrt(3.0 * dt)
    jmax = int(ceil((1 - sqrt(2.0 / 3.0)) / (a * dt)))
    pr = {j: branch(j, a, dt, jmax) for j in range(-jmax, jmax + 1)}
    Q, alpha = [{0: 1.0}], []
    for i in range(n):
        num = sum(q * exp(-j * dx * dt) for j, q in Q[i].items())
        a_i = (log(num) - float(MKT.lnP((i + 1) * dt))) / dt
        alpha.append(a_i)
        nxt = {}
        for j, q in Q[i].items():
            (pu, pm, pd), off = pr[j]
            d = q * exp(-(a_i + j * dx) * dt)
            for p, o in zip((pu, pm, pd), off):
                nxt[j + o] = nxt.get(j + o, 0.0) + p * d
        Q.append(nxt)
    return dx, pr, alpha, Q


dx1, pr1, al1, Q1 = build(A0, SIG0, 1.0, 4)
jmax1 = max(pr1)
show("dx = sigma sqrt(3 dt)", dx1, 0.017321)
show("j_max, the smallest integer >= 0.1835/(a dt)", float(jmax1), 3.0, tol=1e-9)

print()
print("%5s %10s %10s %10s   %s" % ("j", "p_u", "p_m", "p_d", "branches to"))
for j in range(jmax1, -jmax1 - 1, -1):
    (pu, pm, pd), off = pr1[j]
    print("%+5d %10.4f %10.4f %10.4f   %+d, %+d, %+d"
          % (j, pu, pm, pd, j + off[0], j + off[1], j + off[2]))

dx = sigma sqrt(3 dt)                                0.017321   book      0.017321   ok
j_max, the smallest integer >= 0.1835/(a dt)         3.000000   book      3.000000   ok

    j        p_u        p_m        p_d   branches to
   +3     0.8355     0.0891     0.0755   +3, +2, +1
   +2     0.0995     0.6411     0.2595   +3, +2, +1
   +1     0.1299     0.6603     0.2099   +2, +1, +0
   +0     0.1667     0.6667     0.1667   +1, +0, -1
   -1     0.2099     0.6603     0.1299   +0, -1, -2
   -2     0.2595     0.6411     0.0995   -1, -2, -3
   -3     0.0755     0.0891     0.8355   -1, -2, -3


### Stage two: the Arrow–Debreu roll-forward

$Q(i,j)$ is the value today of one unit paid at node $(i,j)$ and nothing elsewhere — the
state prices of Section 13.3, on this lattice. The displacement $\alpha(i)$ is then one
equation in one unknown with everything on the right already computed, which is why the
curve comes back to machine precision rather than to a tolerance.

Three things to notice in the output: the $Q$'s are not symmetric, because the discounting
differs across nodes even where the probabilities mirror; $\alpha$ rises along the curve,
absorbing the forward rates' upward slope; and the bottom node at level 3 carries a
negative rate.

In [16]:
print("%5s" % "j", "".join("%10s" % ("i = %d" % i) for i in range(5)))
for j in range(jmax1, -jmax1 - 1, -1):
    print("%+5d" % j, "".join("%10.4f" % Q1[i][j] if j in Q1[i] else "%10s" % "---"
                              for i in range(5)))
print("%5s" % "alpha", "".join("%9.4f%%" % (100 * al1[i]) if i < len(al1)
                               else "%10s" % "---" for i in range(5)))
print()
print("%5s" % "j", "".join("%10s" % ("i = %d" % i) for i in range(4)))
for j in range(jmax1, -jmax1 - 1, -1):
    print("%+5d" % j, "".join("%9.4f%%" % (100 * (al1[i] + j * dx1)) if j in Q1[i]
                              else "%10s" % "---" for i in range(4)))
print("Short rates r(i,j) = alpha(i) + j dx")
print()
for i in range(4):
    got = sum(Q1[i][j] * exp(-(al1[i] + j * dx1)) for j in Q1[i])
    show("level %d returns P(0,%dy)" % (i, i + 1), got, float(MKT.P(i + 1.0)), tol=1e-12)

    j      i = 0     i = 1     i = 2     i = 3     i = 4
   +3        ---       ---       ---    0.0018    0.0048
   +2        ---       ---    0.0198    0.0369    0.0477
   +1        ---    0.1613    0.2045    0.2092    0.2017
   +0     1.0000    0.6452    0.4799    0.3907    0.3356
   -1        ---    0.1613    0.2081    0.2165    0.2123
   -2        ---       ---    0.0205    0.0395    0.0528
   -3        ---       ---       ---    0.0020    0.0056
alpha    3.2654%   3.6839%   3.9736%   4.1730%       ---

    j      i = 0     i = 1     i = 2     i = 3
   +3        ---       ---       ---   9.3691%
   +2        ---       ---   7.4377%   7.6371%
   +1        ---   5.4159%   5.7056%   5.9050%
   +0    3.2654%   3.6839%   3.9736%   4.1730%
   -1        ---   1.9518%   2.2415%   2.4409%
   -2        ---       ---   0.5095%   0.7089%
   -3        ---       ---       ---  -1.0232%
Short rates r(i,j) = alpha(i) + j dx

level 0 returns P(0,1y)                              0.967873   book    

### The quarter-year tree

The same construction at $\Delta t = 0.25$ over eleven years, as Section 31.8 reports it.
The options that follow are priced at 24 steps a year (`SPY = 24`), a tree that reprices
the curve equally exactly.

In [17]:
SPY = 24
dx4, pr4, al4, Q4 = build(A0, SIG0, 0.25, 44)
show("the quarter-year tree: dx", dx4, 0.008660)
show("the quarter-year tree: j_max", float(max(pr4)), 10.0, tol=1e-9)
worst = max(abs(sum(Q4[i].values()) - float(MKT.P(i * 0.25))) for i in range(45))
print("worst error across all 44 levels: %.3e" % worst)
print("machine precision, not convergence: alpha(i) solves one equation exactly")

the quarter-year tree: dx                            0.008660   book      0.008660   ok
the quarter-year tree: j_max                        10.000000   book     10.000000   ok
worst error across all 44 levels: 1.110e-16
machine precision, not convergence: alpha(i) solves one equation exactly


## 9. Bermudan swaptions — Example 31.8

First check the machinery against a closed form. A European receiver expiring in one year
has a Jamshidian value of 1.1465, and the tree should converge to it.

It does converge, but *not* monotonically. Eight steps a year is worse than
four. Section 20.2 established why: a lattice's error depends on where the nodes fall
relative to the payoff's kink, and refining the grid moves them. A desk that extrapolates
two tree values without checking for this can get a worse answer than either.

In [18]:
PAYS = (2.0, 3.0, 4.0, 5.0, 6.0)


def receiver(a, sigma, spy, exercises, K=0.04, notional=100.0, years=11.0):
    """A receiver swaption on a 4% swap maturing at t = 6, rolled back on the tree.

    A swaption exercised at t buys the payments strictly after t, so the exercise test
    comes before that date's coupon is added."""
    dt = 1.0 / spy
    n = int(round(years / dt))
    dx, pr, alpha, Q = build(a, sigma, dt, n)
    val = {j: 0.0 for j in Q[n]}
    swap = {j: 0.0 for j in Q[n]}
    for i in range(n - 1, -1, -1):
        nv, ns = {}, {}
        for j in Q[i]:
            (pu, pm, pd), off = pr[j]
            d = exp(-(alpha[i] + j * dx) * dt)
            nv[j] = d * sum(p * val.get(j + o, 0.0) for p, o in zip((pu, pm, pd), off))
            ns[j] = d * sum(p * swap.get(j + o, 0.0) for p, o in zip((pu, pm, pd), off))
        t = i * dt
        if any(abs(t - e) < dt / 2 for e in exercises):
            nv = {j: max(nv[j], notional * (ns[j] - 1.0)) for j in nv}
        for tp in PAYS:
            if abs(t - tp) < dt / 2:
                ns = {j: v + K + (1.0 if tp == PAYS[-1] else 0.0) for j, v in ns.items()}
        val, swap = nv, ns
    return val[0]


print("%12s %12s %12s   %s" % ("steps/year", "tree", "error", "book"))
for spy, want in ((4, 1.1637), (8, 1.1696), (12, 1.1667), (24, 1.1586), (48, 1.1502)):
    v = receiver(A0, SIG0, spy, (1.0,))
    flag = "ok" if abs(v - want) < 5e-5 else "DIFFERS"
    print("%12d %12.4f %+12.4f   %s" % (spy, v, v - 1.1465, flag))
print("Jamshidian's closed form for the one-year European is 1.1465")

  steps/year         tree        error   book
           4       1.1637      +0.0172   ok
           8       1.1696      +0.0231   ok
          12       1.1667      +0.0202   ok
          24       1.1586      +0.0121   ok


          48       1.1502      +0.0037   ok
Jamshidian's closed form for the one-year European is 1.1465


### One option with five dates, not five options

Now the product itself. The Bermudan is worth more than any single European, because the
right to choose has value; and far less than their sum, because the rights are substitutes.
A portfolio of five Europeans is five options; a Bermudan is one option with five dates.

In [19]:
EX = (1.0, 2.0, 3.0, 4.0, 5.0)
europeans = [receiver(A0, SIG0, SPY, (e,)) for e in EX]
bermudan = receiver(A0, SIG0, SPY, EX)
print("%-30s %10s" % ("exercise right", "value"))
for e, v in zip(EX, europeans):
    print("%-30s %10.4f" % ("European, %dy only" % e, v))
print("%-30s %10.4f" % ("sum of all five", sum(europeans)))
print("%-30s %10.4f" % ("Bermudan, any one of the five", bermudan))
print()
show("over the best single European, %", 100 * (bermudan / max(europeans) - 1), 42.2, tol=5e-3)
show("as a share of their sum, %", 100 * bermudan / sum(europeans), 37.0, tol=2e-2)

exercise right                      value
European, 1y only                  1.1586
European, 2y only                  1.1953
European, 3y only                  1.0559
European, 4y only                  0.7974
European, 5y only                  0.4412
sum of all five                    4.6484
Bermudan, any one of the five      1.6995

over the best single European, %                    42.181087   book     42.200000   ok
as a share of their sum, %                          36.560480   book     37.000000   ok


## 10. Every European matched, the Bermudan not — Example 31.9

Hold the calibrating European fixed at its tree value and solve for the $\sigma$ that
reproduces it at each mean-reversion rate. Every row prices the calibration instrument
identically, to four decimals, and the Bermudan moves across 13% of its own value.

A single European observes the curve **once**, so it contains no information about $a$ —
and $a$ is what governs how much the curve at year three tells you about year four, which
is exactly what a Bermudan's value depends on.

This cell root-finds a $\sigma$ for each of five mean-reversion rates and rebuilds the tree at
every step, so it is the slowest in the notebook.

In [20]:
target = europeans[0]
print("%6s %10s %12s %12s %10s   %s"
      % ("a", "sigma", "European", "Bermudan", "vs 0.08", "book"))
values = []
for a, want_s, want_b in ((0.01, 0.00823, 1.6474), (0.03, 0.00871, 1.6622),
                          (0.08, 0.01000, 1.6995), (0.15, 0.01202, 1.7541),
                          (0.30, 0.01724, 1.8705)):
    sg = brentq(lambda s: receiver(a, s, SPY, (1.0,)) - target, 0.0005, 0.06, xtol=1e-9)
    b = receiver(a, sg, SPY, EX)
    values.append(b)
    flag = "ok" if abs(sg - want_s) < 5e-6 and abs(b - want_b) < 5e-5 else "DIFFERS"
    print("%6.2f %10.5f %12.4f %12.4f %+9.1f%%   %s"
          % (a, sg, receiver(a, sg, SPY, (1.0,)), b, 100 * (b / bermudan - 1), flag))
print()
show("the Bermudan's span, % of its own value",
     100 * (max(values) - min(values)) / values[2], 13.1, tol=5e-3)

     a      sigma     European     Bermudan    vs 0.08   book


  0.01    0.00823       1.1586       1.6474      -3.1%   ok


  0.03    0.00871       1.1586       1.6622      -2.2%   ok


  0.08    0.01000       1.1586       1.6995      -0.0%   ok


  0.15    0.01202       1.1586       1.7541      +3.2%   ok
  0.30    0.01724       1.1586       1.8705     +10.1%   ok

the Bermudan's span, % of its own value             13.125662   book     13.100000   ok


## 11. Z-spread against option-adjusted spread — Example 31.10

A ten-year 5% bond callable at par from year three, priced at 99. The Z-spread is the
parallel shift that prices it ignoring the call; the OAS is the shift that prices it with
the call modeled. The difference is the call's value at the bond's own spread, expressed as a running yield.

The second table is the warning. The OAS runs from 21 to 72 basis points across parameter
choices that no common calibration has fixed, because it is a *residual* between two nearly equal quantities — so a
modest change in the option's value is a large change in what is left. The Z-spread is a
fact about the price; the OAS is a fact about the price **and** a model.

In [21]:
def callable_bond(a, sigma, spread, callable_=True, cpn=0.05, mat=10, first=3, years=11.0):
    """A ten-year 5% bond, callable at par annually from year three."""
    dt = 1.0 / SPY
    n = int(round(years / dt))
    dx, pr, alpha, Q = build(a, sigma, dt, n)
    v = {j: 0.0 for j in Q[n]}
    for i in range(n - 1, -1, -1):
        nv = {}
        for j in Q[i]:
            (pu, pm, pd), off = pr[j]
            d = exp(-(alpha[i] + j * dx + spread) * dt)
            nv[j] = d * sum(p * v.get(j + o, 0.0) for p, o in zip((pu, pm, pd), off))
        t = i * dt
        if callable_ and any(abs(t - c) < dt / 2 for c in range(first, mat)):
            nv = {j: min(nv[j], 100.0) for j in nv}
        for tp in range(1, mat + 1):
            if abs(t - tp) < dt / 2:
                nv = {j: x + 100.0 * cpn + (100.0 if tp == mat else 0.0)
                      for j, x in nv.items()}
        v = nv
    return v[0]


straight = callable_bond(A0, SIG0, 0.0, callable_=False)
called = callable_bond(A0, SIG0, 0.0, callable_=True)
show("the same bond with the call switched off", straight, 106.9013, tol=5e-6)
show("and with it switched on", called, 101.6743, tol=5e-6)
show("so the embedded call is worth", straight - called, 5.2270, tol=5e-5)
print()

z_sp = brentq(lambda s: callable_bond(A0, SIG0, s, callable_=False) - 99.0,
              -0.02, 0.2, xtol=1e-10)
oas = brentq(lambda s: callable_bond(A0, SIG0, s, callable_=True) - 99.0,
             -0.02, 0.2, xtol=1e-10)
show("Z-spread, bp", 10000 * z_sp, 94.6, tol=5e-4)
show("option-adjusted spread, bp", 10000 * oas, 52.1, tol=5e-4)
show("difference, bp", 10000 * (z_sp - oas), 42.5, tol=2e-3)   # the book subtracts its roundings

print()
print("%8s %10s %12s   %s" % ("a", "sigma", "OAS (bp)", "book"))
for a, sg, want in ((0.03, 0.010, 39.5), (0.08, 0.010, 52.1),
                    (0.20, 0.010, 71.5), (0.08, 0.015, 20.9)):
    oo = brentq(lambda s: callable_bond(a, sg, s, callable_=True) - 99.0,
                -0.02, 0.2, xtol=1e-10)
    flag = "ok" if abs(10000 * oo - want) < 5e-2 else "DIFFERS"
    print("%8.2f %10.3f %12.1f   %s" % (a, sg, 10000 * oo, flag))

the same bond with the call switched off           106.901261   book    106.901300   ok
and with it switched on                            101.674344   book    101.674300   ok
so the embedded call is worth                        5.226917   book      5.227000   ok



Z-spread, bp                                        94.560256   book     94.600000   ok
option-adjusted spread, bp                          52.106552   book     52.100000   ok
difference, bp                                      42.453704   book     42.500000   ok

       a      sigma     OAS (bp)   book


    0.03      0.010         39.5   ok


    0.08      0.010         52.1   ok
    0.20      0.010         71.5   ok


    0.08      0.015         20.9   ok


## 12. What perfect correlation costs — Example 31.11

A five-year option on the spread between the ten-year and two-year swap rates. The spread's
volatility is $\sqrt{\sigma_{10}^2 + \sigma_2^2 - 2\rho\sigma_{10}\sigma_2}$, and a
one-factor model is forced to $\rho = 1$ — the first row.

At $\rho = 1$ the spread volatility is not even a correlation assumption. It is
$|\sigma_{10} - \sigma_2|$, the residual from two rates moving in lockstep at slightly
different speeds. No calibration repairs this, and the error always has the same sign,
because $\rho = 1$ *minimizes* the spread's volatility.

In [22]:
F10, F2, S10, S2, ANN, T_CMS = 0.0340, 0.0280, 0.0090, 0.0075, 4.20, 5.0
show("the forward spread, bp", 10000 * (F10 - F2), 60.0, tol=5e-4)
show("at rho = 1 the spread vol is |sigma_10 - sigma_2|, bp",
     10000 * abs(S10 - S2), 15.0, tol=5e-4)

print()
print("%8s %14s %16s %8s   %s"
      % ("rho", "spread vol", "option value", "ratio", "book"))
base = None
for rho, w_vol, w_val in ((1.00, 15.0, 56.2), (0.95, 30.0, 112.4), (0.90, 39.7, 148.7),
                          (0.80, 54.1, 202.6), (0.70, 65.4, 245.0), (0.50, 83.5, 312.9)):
    vol = sqrt(S10 ** 2 + S2 ** 2 - 2 * rho * S10 * S2)
    value = ANN * vol * sqrt(T_CMS / (2 * pi))      # Bachelier, struck at the money
    base = base or value
    flag = "ok" if abs(10000 * vol - w_vol) < 5e-2 and abs(10000 * value - w_val) < 5e-2 \
        else "DIFFERS"
    print("%8.2f %11.1f bp %13.1f bp %8.2f   %s"
          % (rho, 10000 * vol, 10000 * value, value / base, flag))

the forward spread, bp                              60.000000   book     60.000000   ok
at rho = 1 the spread vol is |sigma_10 - sigma_2|, bp      15.000000   book     15.000000   ok

     rho     spread vol     option value    ratio   book
    1.00        15.0 bp          56.2 bp     1.00   ok
    0.95        30.0 bp         112.4 bp     2.00   ok
    0.90        39.7 bp         148.7 bp     2.65   ok
    0.80        54.1 bp         202.6 bp     3.61   ok
    0.70        65.4 bp         245.0 bp     4.36   ok
    0.50        83.5 bp         312.9 bp     5.57   ok


## 13. How often the model says rates are below zero — Example 31.12

Gaussian short rates go negative with positive probability, and at ten years the middle row
assigns better than one chance in five to it.

Before rates went negative (Denmark in 2012, the euro area in 2014) this was cited as the model's principal flaw. Then they did, and
stayed there, and the property became the reason Hull–White survived while its positive
competitors could not be calibrated at all without a shift. A model's implausible region is where its next
test comes from.

In [23]:
print("%-26s %8s %8s %8s %8s   %s" % ("", "1y", "2y", "5y", "10y", "book"))
for a, sg, want in ((0.05, 0.010, (0.1, 1.3, 6.5, 11.6)),
                    (0.05, 0.015, (2.0, 6.9, 15.7, 21.3)),
                    (0.10, 0.010, (0.1, 1.0, 4.6, 7.5))):
    got = []
    for T in (1.0, 2.0, 5.0, 10.0):
        sd = sqrt(sg ** 2 / (2 * a) * (1 - exp(-2 * a * T)))
        got.append(100 * N(-0.03 / sd))
    flag = "ok" if max(abs(g - w) for g, w in zip(got, want)) < 5e-2 else "DIFFERS"
    print("%-26s %7.1f%% %7.1f%% %7.1f%% %7.1f%%   %s"
          % ("a = %.2f, sigma = %.1f%%" % (a, 100 * sg), *got, flag))
print("Probability that Hull-White's r(T) is below zero, on a flat 3% forward curve")

                                 1y       2y       5y      10y   book
a = 0.05, sigma = 1.0%         0.1%     1.3%     6.5%    11.6%   ok
a = 0.05, sigma = 1.5%         2.0%     6.9%    15.7%    21.3%   ok
a = 0.10, sigma = 1.0%         0.1%     1.0%     4.6%     7.5%   ok
Probability that Hull-White's r(T) is below zero, on a flat 3% forward curve
